In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
from langchain_core.tools import tool


In [ ]:
from langchain_tavily import TavilySearch
tavily_search = TavilySearch(
    max_results=2,
    topic="general",
    include_answer=True
)
@tool
def web_search(query:str) -> str:

    """
    Uses tavily tool to search the web for up-to-date information and returns relevant results with sources.
    """

    return tavily_search.invoke({"query":query})['answer']
# web_search("what is recent ai news")

In [ ]:
web_search.invoke("what are guardrails")

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
    # model should always return structured output
)


In [ ]:
llm.invoke("hi")

### Input Context

#### System Prompt

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import StateBackend

# Files are saved in langgraph state

# By default we provide a StateBackend
agent = create_deep_agent(
    model=llm,
    system_prompt=(
        "You are a reearch assistant specialising in scientific literature."
        "Always cite sources. Use subagents for parallel research on different topics."
    )
    )



In [ ]:
agent

##### Memory
###### Final prompt sent to LLM - 
###### System prompt(static and tells tone, behaviour) + memory files + user query

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
checkpointer = MemorySaver()

In [ ]:
from deepagents import create_deep_agent
# from deepagents.backends import StateBackend

# Files are saved in langgraph state

# By default we provide a StateBackend
agent = create_deep_agent(
    model=llm,
    checkpointer=checkpointer
    )



In [ ]:
agent

In [ ]:
from deepagents import create_deep_agent
# from deepagents.backends import StateBackend

# Files are saved in langgraph state

# By default we provide a StateBackend
agent = create_deep_agent(
    model=llm,
    memory=["projects/AGENTS.md"],
    checkpointer=checkpointer,

    )



In [ ]:
agents_md="""
# Agent Profile & Memory

This file gives the agent background knowledge about its own architectire, the project conventions and how it should operate.

## Project Overview

This repository contains an AI application built with LangGraph and FastAPI.

The primary goals are:
- Write clean, maintainable code.
- Prefer simple solutions over complex ones.
- Keep changes minimal and focused.
- Preserve existing behavior unless explicitly requested.

---

## Coding Standards

- Follow PEP 8.
- Use type hints whenever possible.
- Write descriptive variable and function names.
- Keep functions under 50 lines when practical.
- Avoid duplicate code.

---

## File Organization

- API routes belong in `app/api/`
- Business logic belongs in `app/services/`
- Utility functions belong in `app/utils/`
- Database code belongs in `app/db/`

Do not create new top-level directories unless necessary.

---

## Python Guidelines

- Use pathlib instead of os.path.
- Prefer dataclasses or Pydantic models.
- Avoid global variables.
- Use logging instead of print().
- Raise meaningful exceptions.

---

## Testing

Whenever modifying code:

- Update existing tests if needed.
- Add tests for new functionality.
- Run the relevant tests before finishing.
- Do not remove tests unless requested.

---

## Dependencies

Prefer existing libraries already used in the project.

Do not introduce new dependencies unless they provide significant value.

---

## Git

Do not modify:

- .gitignore
- LICENSE
- README.md

unless explicitly instructed.

---

## Security

Never:

- Hardcode API keys
- Commit secrets
- Disable authentication
- Log passwords or tokens

Use environment variables for secrets.

---

## Documentation

When adding a new module:

- Add a module docstring.
- Document public functions.
- Update README only if requested.

---

## Error Handling

Prefer explicit exception handling.

Return informative error messages while avoiding leakage of sensitive information.

---

## Performance

Avoid unnecessary database queries.

Avoid loading large files entirely into memory.

Prefer generators when processing large datasets.

---

## Before Finishing

Verify:

- Code compiles.
- Imports are correct.
- No unused variables.
- No obvious bugs.
- Formatting is clean.

Provide a brief summary of the changes made.
"""

In [ ]:
from deepagents.backends.utils import create_file_data
create_file_data(agents_md)

In [ ]:
result=agent.invoke({
    "messages":[
        {
            "role":"user",
            "content":"What is in your memory?"
        }
       
    ],
     "files":{"projects/AGENTS.md":create_file_data(agents_md)}
},
config={"configurable":{"thread_id":"2"}}
)

In [ ]:
result

In [ ]:
from deepagents.backends.store import StoreBackend
from langgraph.store.memory import InMemoryStore

store=InMemoryStore()

store.put(("memories"),"AGENTS.md", create_file_data(agents_md))

store_agent=create_deep_agent(
    model=llm,
    backend=StoreBackend(store=store,namespace=lambda rt:("memories")),
    memory=["/projects/AGENTS.md"],
    store=store,
    checkpointer=checkpointer
)

In [ ]:
result=store_agent.invoke({
     "messages":[
            {
                "role":"user",
                "content":"What is in your memory?"
            }
           
        ]
    },
    config={"configurable":{"thread_id":"2"}}
    
)

In [ ]:
result

In [ ]:
from deepagents.backends.filesystem import FilesystemBackend

fs_agent = create_deep_agent(
    model=llm,
    backend = FilesystemBackend(root_dir="projects", virtual_mode=True),
    memory=["AGENTS.md"],
    checkpointer=checkpointer
)

In [ ]:
result=fs_agent.invoke({
      "messages":[
                {
                    "role":"user",
                    "content":"Do you have a memory?"
                }
               
            ]
        },
        config={"configurable":{"thread_id":"2"}}
)

In [ ]:
result

In [ ]:
# from urllib.request import urlopen
# from pathlib import Path
# from langchain_quickjs import CodeInterpreterMiddleware

In [ ]:
checkpointer=MemorySaver()
backend=StateBackend()

skill_dirs=["langgraph","python","aws","report-writer"]
skills_files={
    f"/skills/{name}/skill.md":create_file_data(
        Path( f"skills/{name}/skill.md").read_text(encoding="utf-8")
    )

    for name in skill_dirs
}

In [ ]:
skills_files

In [ ]:
agent=create_deep_agent(
    model=llm,
    backend=backend,
    skills=["/skills/"],
    checkpointer=checkpointer,
    system_prompt="Make use of relevant skills and memory if provided and if needed"
)

In [ ]:
result=agent.invoke({
    "messages":[
        {
            "role":"user",
            "content":"What skills do you have available?"
        }
       
    ],
     "files":skills_files
},
config={"configurable":{"thread_id":"1"}}
)

In [ ]:
result

In [ ]:
print(result["messages"][-1].content)

In [ ]:
result=agent.invoke({
    "messages":[
        {
            "role":"user",
            "content":"Create a workflow in langgraph."
        }
       
    ],
     "files":skills_files
},
config={"configurable":{"thread_id":"4"}}
)

In [ ]:
for m in result["messages"]:
    m.pretty_print()

### Subagents turn one generalist agent into a team of specialists

In [ ]:
research_subagent={
    "name":"research-subagent",
   "description":"ALWAYS use this subagent whenever the user asks to research,investigate, compare, summarize information from multiple sources, or requires web search.",
   "system_prompt":"You are a great researcher",
   "tools": [web_search],
   "model":llm
}

In [ ]:
subagents=[research_subagent]

In [ ]:
agent=create_deep_agent(
    model=llm,
    subagents=subagents,
    system_prompt="Always ake use of subagents for the tasks they are delegated for."
)

In [ ]:
# result=agent.invoke({
#     "messages":[
#         {
#             "role":"user",
#             "content":"Research about Guardrails and give me a detailed summary."
#         }
       
#     ]
#     #  "files":skills_files
# },
# config={"configurable":{"thread_id":"500"}}
# )

In [ ]:
for m in result["messages"]:
    m.pretty_print()

In [ ]:
from pydantic import BaseModel, Field

class ResearchFinding(BaseModel):
    """Structured findings from a research task"""

    summary:str=Field(description="Summary of findings")
    confidence:float=Field(description="Confidence score from 0 to 1")
    sources:list[str]=Field(description="List of source urls")

    

In [ ]:
research_subagent={
    "name":"research-subagent",
   "description":"ALWAYS use this subagent whenever the user asks to research,investigate, compare, summarize information from multiple sources, or requires web search.",
   "system_prompt":"You are a great researcher",
   "tools": [web_search],
   "model":llm,
   "response_format":ResearchFinding,
#    "files":research_files
}

# agents_md = {
#     "AGENTS.md": create_file_data(
#         Path("AGENTS.md").read_text(encoding="utf-8")
#     )
# }

# research_files = {
#     **agents_md,
#     "skills/research/skill.md": create_file_data(
#         Path("skills/research/skill.md").read_text(encoding="utf-8")
#     ),
# }


In [ ]:
subagents=[research_subagent]

In [ ]:
agent=create_deep_agent(
    model=llm,
    subagents=subagents,
    system_prompt="Gather the output of all the subagents as needed and give a summarised response on the user query."
)

In [ ]:
# result=agent.invoke({
#     "messages":[
#         {
#             "role":"user",
#             "content":"Research about deepfakes and thereaafter give a summary on the topic."
#         }
       
#     ]
#     #  "files":skills_f iles
# },
# config={"configurable":{"thread_id":"111"}}
# )

In [ ]:
for m in result["messages"]:
    m.pretty_print()